# Baum-Welch

From Wikipedia (https://en.wikipedia.org/wiki/Baum%E2%80%93Welch_algorithm): 

Baum–Welch algorithm is a special case of the expectation–maximization algorithm used to find the unknown parameters of a hidden Markov model (HMM).

It makes use of the forward-backward algorithm to compute the statistics for the expectation step

The Baum–Welch algorithm, the primary method for inference in hidden Markov models, is numerically unstable due to its recursive calculation of joint probabilities. As the number of variables grows, these joint probabilities become increasingly small, leading to the forward recursions rapidly approaching values below machine precision.

---

The Baum–Welch algorithm uses the well known EM algorithm to find the maximum likelihood estimate of the parameters of a hidden Markov model given a set of observed feature vectors.

Thus we can describe a hidden Markov model by $\theta = (A, B, \pi)$. The Baum–Welch algorithm finds a local maximum for:

$$\theta^* = \underset{\theta}{\arg\max} \ P(Y \mid \theta)$$

i.e. the HMM parameters $\theta$ that maximize the probability of the observations.

## Algorithm

Set $\theta = (A, B, \pi)$ with random initial conditions. They can also be set using prior information about the parameters if available; this can speed up the algorithm and steer it toward the desired local maximum.

### Forward Procedure

Let $\alpha_i(t) = P(Y_1 = y_1, \ldots, Y_t = y_t, X_t = i \mid \theta)$, the probability of seeing the observations $y_1, y_2, \ldots, y_t$ and being in state $i$ at time $t$. This is found recursively:

1. $\alpha_i(1) = \pi_i b_i(y_1)$
2. $\alpha_i(t+1) = b_i(y_{t+1}) \sum_{j=1}^{N} \alpha_j(t) a_{ji}$

Since this series converges exponentially to zero, the algorithm will numerically underflow for longer sequences. However, this can be avoided by scaling $\alpha$ in the forward procedure and $\beta$ in the backward procedure.

### Backward Procedure

Let $\beta_i(t) = P(Y_{t+1} = y_{t+1}, \ldots, Y_T = y_T \mid X_t = i, \theta)$, the probability of the ending partial sequence $y_{t+1}, \ldots, y_T$ given starting state $i$ at time $t$. We calculate $\beta_i(t)$ as:

1. $\beta_i(T) = 1$
2. $\beta_i(t) = \sum_{j=1}^{N} \beta_j(t+1) a_{ij} b_j(y_{t+1})$

### Update

We can now calculate the temporary variables. According to Bayes' theorem:

$$\gamma_i(t) = P(X_t = i \mid Y, \theta) = \frac{P(X_t = i, Y \mid \theta)}{P(Y \mid \theta)} = \frac{\alpha_i(t)\beta_i(t)}{\sum_{j=1}^{N} \alpha_j(t)\beta_j(t)}$$

which is the probability of being in state $i$ at time $t$ given the observed sequence $Y$ and the parameters $\theta$.

$$\xi_{ij}(t) = P(X_t = i, X_{t+1} = j \mid Y, \theta) = \frac{P(X_t = i, X_{t+1} = j, Y \mid \theta)}{P(Y \mid \theta)} = \frac{\alpha_i(t) a_{ij} \beta_j(t+1) b_j(y_{t+1})}{\sum_{k=1}^{N} \sum_{w=1}^{N} \alpha_k(t) a_{kw} \beta_w(t+1) b_w(y_{t+1})}$$

which is the probability of being in state $i$ and $j$ at times $t$ and $t+1$ respectively, given the observed sequence $Y$ and parameters $\theta$.

The denominators of $\gamma_i(t)$ and $\xi_{ij}(t)$ are the same; they represent the probability of making the observation $Y$ given the parameters $\theta$.

The parameters of the hidden Markov model $\theta$ can now be updated:

$$\pi_i^* = \gamma_i(1)$$

which is the expected frequency spent in state $i$ at time $1$.

$$a_{ij}^* = \frac{\sum_{t=1}^{T-1} \xi_{ij}(t)}{\sum_{t=1}^{T-1} \gamma_i(t)}$$

which is the expected number of transitions from state $i$ to state $j$ compared to the expected total number of transitions starting in state $i$, including from state $i$ to itself.

$$b_i^*(v_k) = \frac{\sum_{t=1}^{T} \mathbf{1}_{y_t = v_k} \gamma_i(t)}{\sum_{t=1}^{T} \gamma_i(t)}$$

where

$$\mathbf{1}_{y_t = v_k} = \begin{cases} 1 & \text{if } y_t = v_k \\ 0 & \text{otherwise} \end{cases}$$

is an indicator function, and $b_i^*(v_k)$ is the expected number of times the output observations have been equal to $v_k$ while in state $i$, over the expected total number of times in state $i$.

These steps are repeated iteratively until a desired level of convergence.

> **Note:** It is possible to over-fit a particular dataset, i.e. $P(Y \mid \theta_{\text{final}}) > P(Y \mid \theta_{\text{true}})$. The algorithm also does not guarantee a global maximum.

### Multiple Sequences

The algorithm described thus far assumes a single observed sequence $Y = y_1, \ldots, y_T$. However, in many situations there are several observed sequences $Y_1, \ldots, Y_R$. In this case, the information from all observed sequences must be used in the parameter updates. Assuming $\gamma_{ir}(t)$ and $\xi_{ijr}(t)$ have been computed for each sequence, the parameters are updated as:

$$\pi_i^* = \frac{\sum_{r=1}^{R} \gamma_{ir}(1)}{R}$$

$$a_{ij}^* = \frac{\sum_{r=1}^{R} \sum_{t=1}^{T-1} \xi_{ijr}(t)}{\sum_{r=1}^{R} \sum_{t=1}^{T-1} \gamma_{ir}(t)}$$

$$b_i^*(v_k) = \frac{\sum_{r=1}^{R} \sum_{t=1}^{T} \mathbf{1}_{y_{tr} = v_k} \gamma_{ir}(t)}{\sum_{r=1}^{R} \sum_{t=1}^{T} \gamma_{ir}(t)}$$

where

$$\mathbf{1}_{y_{tr} = v_k} = \begin{cases} 1 & \text{if } y_{t,r} = v_k \\ 0 & \text{otherwise} \end{cases}$$

is an indicator function.

## Imports

In [1]:
import numpy as np

## Functions

In [2]:
def process_symbols(symbols):
    """
    Returns a sorted array of unique symbols from the input string.
    """
    return np.array(sorted(set(symbols)))

def encode(sequence,alphabet):
        """
        Encodes a sequence of characters into a list of integer indices
        based on their position in the alphabet.

        Parameters
        ----------
        sequence : str
            The input sequence to encode (e.g. an amino acid sequence "ACMKL").

        Returns
        -------
        list of int
            A list of integers where each element is the index of the
            corresponding character in the alphabet.

        Example
        -------
        >>> hmm = HMM(n_states=3, alphabet="ACDEFGHIKLMNPQRSTVWY")
        >>> hmm.encode("ACD")
        [0, 1, 2]
        """

        char_to_idx = {char: i for i, char in enumerate(alphabet)}

        try:
            enc = [char_to_idx[char] for char in sequence]
        except KeyError as e:
            raise ValueError(f"Sequence contains character not found in alphabet: {e}")

        return enc

## HMM Implementation

In [3]:
class HMM():
    """
    Hidden Markov Model (HMM) implementation with Baum-Welch training
    and Viterbi decoding.

    Attributes
    ----------
    n_states : int
        Number of hidden states in the model.
    alphabet : str
        String of valid emission symbols (e.g. "ACDEFGHIKLMNPQRSTVWY").
    n_symbols : int
        Number of unique symbols in the alphabet.
    pi : np.ndarray of shape (n_states,)
        Initial state probability vector.
    A : np.ndarray of shape (n_states, n_states)
        Transition probability matrix, where A[i][j] is the probability
        of transitioning from state i to state j.
    B : np.ndarray of shape (n_states, n_symbols)
        Emission probability matrix, where B[i][k] is the probability
        of emitting symbol k in state i.
    alpha : np.ndarray of shape (n_states, sequence_length)
        The scaled forward probabilities.
    G_alpha : np.ndarray of shape (sequence_length,)
        The scaling factors at each position t of the forward algorithm.
    beta : np.ndarray of shape (n_states, sequence_length)
        The scaled backward probabilities.
    G_beta : np.ndarray of shape (sequence_length,)
        The scaling factors at each position t of the backward algorithm.
    """

    def __init__(self, number_of_states, alphabet, seed=42):
        """
        Initializes the HMM with random parameters.

        Parameters
        ----------
        number_of_states : int
            Number of hidden states in the model.
        alphabet : str
            String of valid emission symbols.
        seed : int, optional
            Random seed for reproducibility (default: 42).
        """

        np.random.seed(seed)

        self.n_states = number_of_states
        self.alphabet = alphabet
        self.n_symbols = len(set(alphabet))

        # Initialize and normalize pi
        self.pi = np.random.dirichlet(np.ones(self.n_states))

        # Initialize and normalize transition matrix A
        self.A = np.random.dirichlet(np.ones(self.n_states), size=self.n_states)

        # Initialize and normalize emission matrix B
        self.B = np.random.dirichlet(np.ones(self.n_symbols), size=self.n_states)

        # Forward algorithm results
        self.alpha = None
        self.G_alpha = None

        # Backward algorithm results
        self.beta = None
        self.G_beta = None

        # Baum-Welch training
        self.gamma = None
        self.xi = None

    def _initialize_forward(self, input_encode):
        """
        Initializes the forward algorithm by computing the scaled alpha values
        for the first observation.

        The alpha matrix stores the probability of being in a given state at each
        position, given all previous observations. At t=0, this is computed as:

            alpha[i][0] = pi[i] * B[i][y_0]

        The values are then scaled by G to avoid numerical underflow.
        Results are stored in self.alpha and self.G_alpha.

        Parameters
        ----------
        input_encode : list of int
            The encoded input sequence (output of encode()).
        """

        self.alpha = np.zeros(shape=(self.n_states, len(input_encode)))
        self.G_alpha = np.zeros(len(input_encode))
        G = 0

        for i in range(self.n_states):
            self.alpha[i][0] = self.pi[i] * self.B[i][input_encode[0]]
            G += self.alpha[i][0]

        for i in range(self.n_states):
            self.alpha[i][0] /= G

        self.G_alpha[0] = G

    def forward(self, input_encode):
        """
        Runs the forward algorithm for positions t=1 to T-1,
        filling in the alpha matrix with scaled forward probabilities.
        Calls _initialize_forward() internally.

        Parameters
        ----------
        input_encode : list of int
            The encoded input sequence (output of encode()).
        """

        self._initialize_forward(input_encode)

        for t in range(1, len(input_encode)):
            G = 0
            for j in range(self.n_states):
                _sum = 0
                for k in range(self.n_states):
                    _sum += self.alpha[k][t-1] * self.A[k][j]
                self.alpha[j][t] = self.B[j][input_encode[t]] * _sum
                G += self.alpha[j][t]

            for j in range(self.n_states):
                self.alpha[j][t] /= G

            self.G_alpha[t] = G

        for t in range(1, len(input_encode)):
            # matrix multiply: (n_states,) = (n_states, n_states).T @ (n_states,)
            self.alpha[:, t] = self.B[:, input_encode[t]] * (self.A.T @ self.alpha[:, t-1])
            G = self.alpha[:, t].sum()
            self.alpha[:, t] /= G
            self.G_alpha[t] = G

    def _initialize_backward(self, input_encode):
        """
        Initializes the backward algorithm by setting beta values to 1
        at the last position T-1, as per the standard HMM definition:

            beta[i][T-1] = 1  for all states i

        Results are stored in self.beta and self.G_beta is reset.

        Parameters
        ----------
        input_encode : list of int
            The encoded input sequence (output of encode()).
        """

        self.beta = np.zeros(shape=(self.n_states, len(input_encode)))
        self.G_beta = np.zeros(len(input_encode))

        for i in range(self.n_states):
            self.beta[i][-1] = 1

    def backward(self, input_encode):
        """
        Runs the backward algorithm for positions t=T-2 to 0,
        filling in the beta matrix with scaled backward probabilities.
        Calls _initialize_backward() internally.

        Parameters
        ----------
        input_encode : list of int
            The encoded input sequence (output of encode()).
        """

        self._initialize_backward(input_encode)

        for t in range(len(input_encode) - 2, -1, -1):
            G = 0
            for j in range(self.n_states):
                _sum = 0
                for k in range(self.n_states):
                    _sum += self.B[k][input_encode[t+1]] * self.A[j][k] * self.beta[k][t+1]
                self.beta[j][t] = _sum
                G += self.beta[j][t]

            for j in range(self.n_states):
                self.beta[j][t] /= G

            self.G_beta[t] = G

    def compute_gamma(self):
        """
        Computes the posterior probabilities (gamma) for each state at each
        position in the sequence.

        gamma[i][t] is the probability of being in state i at time t given
        the full observation sequence:

            gamma[i][t] = (alpha[i][t] * beta[i][t]) / sum_j(alpha[j][t] * beta[j][t])

        Results are stored in self.gamma.
        Must call forward() and backward() before this method.
        """

        T = self.alpha.shape[1]
        self.gamma = np.zeros(shape=(self.n_states, T), dtype=float)

        for t in range(T):
            norm_factor = 0
            for j in range(self.n_states):
                norm_factor += self.alpha[j][t] * self.beta[j][t]

            for j in range(self.n_states):
                self.gamma[j][t] = (self.alpha[j][t] * self.beta[j][t]) / norm_factor


    def compute_xi(self, input_encode):
        """
        Computes the pairwise posterior probabilities (xi) for each pair of
        states at each position in the sequence.

        xi[i][j][t] is the probability of being in state i at time t and
        transitioning to state j at time t+1, given the full observation sequence:

            xi[i][j][t] = (alpha[i][t] * A[i][j] * B[j][y_{t+1}] * beta[j][t+1]) /
                           sum_i sum_j (alpha[i][t] * A[i][j] * B[j][y_{t+1}] * beta[j][t+1])

        Results are stored in self.xi.
        Must call forward() and backward() before this method.

        Parameters
        ----------
        input_encode : list of int
        The encoded input sequence (output of encode()).
        """

        T = self.alpha.shape[1]
        self.xi = np.zeros(shape=(self.n_states, self.n_states, T - 1), dtype=float)

        for t in range(T - 1):
            norm_xi = 0
            for j in range(self.n_states):
                for k in range(self.n_states):
                    self.xi[j][k][t] = (self.alpha[j][t] * self.A[j][k] *
                                        self.B[k][input_encode[t+1]] * self.beta[k][t+1])
                    norm_xi += self.xi[j][k][t]

            for j in range(self.n_states):
                for k in range(self.n_states):
                    self.xi[j][k][t] /= norm_xi

    def _update_parameters(self, all_gamma, all_xi, all_encoded):
        """
        Updates the HMM parameters pi, A, and B using the Baum-Welch
        update equations across multiple sequences.

        Parameters
        ----------
        all_gamma : list of np.ndarray
            List of gamma matrices, one per sequence.
        all_xi : list of np.ndarray
            List of xi arrays, one per sequence.
        all_encoded : list of list of int
            List of encoded sequences.
        """

        R = len(all_encoded)

        # Update pi: average of gamma at t=0 across all sequences
        self.pi = np.zeros(self.n_states)
        for r in range(R):
            for i in range(self.n_states):
                self.pi[i] += all_gamma[r][i][0]
        self.pi /= R

        # Update A: sum of xi / sum of gamma across all sequences and time steps
        self.A = np.zeros(shape=(self.n_states, self.n_states))
        for r in range(R):
            T = len(all_encoded[r])
            for i in range(self.n_states):
                for j in range(self.n_states):
                    self.A[i][j] += np.sum(all_xi[r][i][j])
    
        # Normalize each row
        for i in range(self.n_states):
            self.A[i] /= np.sum(self.A[i])

        # Update B: expected emissions per state
        self.B = np.zeros(shape=(self.n_states, self.n_symbols))
        for r in range(R):
            T = len(all_encoded[r])
            for t in range(T):
                for i in range(self.n_states):
                    self.B[i][all_encoded[r][t]] += all_gamma[r][i][t]

        # Normalize each row
        for i in range(self.n_states):
            self.B[i] /= np.sum(self.B[i])

    def baum_welch(self, sequences, max_iter=100, epsilon=1e-4):
        """
        Trains the HMM using the Baum-Welch algorithm over multiple sequences.

        Parameters
        ----------
        sequences : list of str
            List of observed sequences to train on.
        max_iter : int, optional
            Maximum number of iterations (default: 100).
        epsilon : float, optional
            Convergence threshold for log-likelihood (default: 1e-4).

        Returns
        -------
        log_likelihoods : list of float
            Log-likelihood at each iteration, useful for diagnostics.
        """

        # Encode all sequences once
        all_encoded = sequences
        log_likelihoods = []

        for iteration in range(max_iter):

            all_gamma = []
            all_xi = []
            log_likelihood = 0
    
            # E-step: forward, backward, gamma, xi for each sequence
            for enc in all_encoded:
                self.forward(enc)
                self.backward(enc)
                self.compute_gamma()
                self.compute_xi(enc)

                all_gamma.append(self.gamma.copy())
                all_xi.append(self.xi.copy())
                log_likelihood += np.sum(np.log(self.G_alpha))

            log_likelihoods.append(log_likelihood)
            print(f"Iteration {iteration + 1} | log-likelihood: {log_likelihood:.4f}")

            # Check convergence
            if iteration > 0:
                if abs(log_likelihoods[-1] - log_likelihoods[-2]) < epsilon:
                    print(f"Converged at iteration {iteration + 1}")
                    break

            # M-step: update parameters
            self._update_parameters(all_gamma, all_xi, all_encoded)

        return log_likelihoods

## HMM Efficient Implementation

Some small differences compared to the previous version. Some vectorization instead of loops

In [4]:
class HMM_efficient():
    """
    Hidden Markov Model (HMM) implementation with Baum-Welch training
    and Viterbi decoding.

    Attributes
    ----------
    n_states : int
        Number of hidden states in the model.
    alphabet : str
        String of valid emission symbols (e.g. "ACDEFGHIKLMNPQRSTVWY").
    n_symbols : int
        Number of unique symbols in the alphabet.
    pi : np.ndarray of shape (n_states,)
        Initial state probability vector.
    A : np.ndarray of shape (n_states, n_states)
        Transition probability matrix, where A[i][j] is the probability
        of transitioning from state i to state j.
    B : np.ndarray of shape (n_states, n_symbols)
        Emission probability matrix, where B[i][k] is the probability
        of emitting symbol k in state i.
    alpha : np.ndarray of shape (n_states, sequence_length)
        The scaled forward probabilities.
    G_alpha : np.ndarray of shape (sequence_length,)
        The scaling factors at each position t of the forward algorithm.
    beta : np.ndarray of shape (n_states, sequence_length)
        The scaled backward probabilities.
    G_beta : np.ndarray of shape (sequence_length,)
        The scaling factors at each position t of the backward algorithm.
    """

    def __init__(self, number_of_states, alphabet, seed=42):
        """
        Initializes the HMM with random parameters.

        Parameters
        ----------
        number_of_states : int
            Number of hidden states in the model.
        alphabet : str
            String of valid emission symbols.
        seed : int, optional
            Random seed for reproducibility (default: 42).
        """

        np.random.seed(seed)

        self.n_states = number_of_states
        self.alphabet = alphabet
        self.n_symbols = len(set(alphabet))

        # Initialize and normalize pi
        self.pi = np.random.dirichlet(np.ones(self.n_states))

        # Initialize and normalize transition matrix A
        self.A = np.random.dirichlet(np.ones(self.n_states), size=self.n_states)

        # Initialize and normalize emission matrix B
        self.B = np.random.dirichlet(np.ones(self.n_symbols), size=self.n_states)

        # Forward algorithm results
        self.alpha = None
        self.G_alpha = None

        # Backward algorithm results
        self.beta = None
        self.G_beta = None

        # Baum-Welch training
        self.gamma = None
        self.xi = None

    def forward(self, input_encode):
        """
        Runs the forward algorithm, filling the alpha matrix with scaled
        forward probabilities using the recurrence:

            alpha[:, 0] = pi * B[:, y_0]
            alpha[:, t] = B[:, y_t] * (A.T @ alpha[:, t-1])

        All columns are scaled by G at each step to avoid numerical underflow.
        Results are stored in self.alpha and self.G_alpha.

        Parameters
        ----------
        input_encode : list of int
            The encoded input sequence (output of encode()).
        """

        T = len(input_encode)
        self.alpha = np.zeros((self.n_states, T))
        self.G_alpha = np.zeros(T)

        # t=0
        self.alpha[:, 0] = self.pi * self.B[:, input_encode[0]]
        self.G_alpha[0] = self.alpha[:, 0].sum()
        self.alpha[:, 0] /= self.G_alpha[0]

        # t=1 to T-1
        for t in range(1, T):
            self.alpha[:, t] = self.B[:, input_encode[t]] * (self.A.T @ self.alpha[:, t-1])
            # Rescaling (divide by sum of values of all states in current t)
            self.G_alpha[t] = self.alpha[:, t].sum()
            self.alpha[:, t] /= self.G_alpha[t]

    def backward(self, input_encode):
        """
        Runs the backward algorithm, filling the beta matrix with scaled
        backward probabilities using the recurrence:

            beta[:, T-1] = 1
            beta[:, t]   = A @ (B[:, y_{t+1}] * beta[:, t+1])

        All columns are scaled by G at each step to avoid numerical underflow.
        Results are stored in self.beta and self.G_beta.

        Parameters
        ----------
        input_encode : list of int
            The encoded input sequence (output of encode()).
        """

        T = len(input_encode)
        self.beta = np.zeros((self.n_states, T))
        self.G_beta = np.zeros(T)

        # t=T-1: boundary condition
        self.beta[:, -1] = 1

        # t=T-2 to 0
        for t in range(T - 2, -1, -1):
            self.beta[:, t] = self.A @ (self.B[:, input_encode[t+1]] * self.beta[:, t+1])
            # Rescaling (divide by sum of values of all states in current t)
            self.G_beta[t] = self.beta[:, t].sum()
            self.beta[:, t] /= self.G_beta[t]

    def compute_gamma(self):
        """
        Computes the posterior probabilities (gamma) for each state at each
        position in the sequence.

        gamma[i][t] is the probability of being in state i at time t given
        the full observation sequence:

            gamma[:, t] = (alpha[:, t] * beta[:, t]) / sum_j(alpha[j][t] * beta[j][t])

        Results are stored in self.gamma.
        Must call forward() and backward() before this method.
        """
        
        # Brand new vector created just pairwise multipilication
        self.gamma = self.alpha * self.beta
        self.gamma /= self.gamma.sum(axis=0, keepdims=True)


    def compute_xi(self, input_encode):
        """
        Computes the pairwise posterior probabilities (xi) for each pair of
        states at each position in the sequence.

        xi[i][j][t] is the probability of being in state i at time t and
        transitioning to state j at time t+1, given the full observation sequence:

            xi[:, :, t] = outer(alpha[:, t], B[:, y_{t+1}] * beta[:, t+1]) * A
                          normalized by its sum

        Results are stored in self.xi.
        Must call forward() and backward() before this method.

        Parameters
        ----------
        input_encode : list of int
            The encoded input sequence (output of encode()).
        """

        T = self.alpha.shape[1]
        self.xi = np.zeros((self.n_states, self.n_states, T - 1))

        for t in range(T - 1):
            # outer product: alpha[:, t] (n_states,) x (B[:, y_{t+1}] * beta[:, t+1]) (n_states,)
            self.xi[:, :, t] = np.outer(self.alpha[:, t], self.B[:, input_encode[t+1]] * self.beta[:, t+1]) * self.A
            self.xi[:, :, t] /= self.xi[:, :, t].sum()

    def _update_parameters(self, all_gamma, all_xi, all_encoded):
        """
        Updates the HMM parameters pi, A, and B using the Baum-Welch
        update equations across multiple sequences.

        Parameters
        ----------
        all_gamma : list of np.ndarray
            List of gamma matrices, one per sequence.
        all_xi : list of np.ndarray
            List of xi arrays, one per sequence.
        all_encoded : list of list of int
            List of encoded sequences.
        """

        R = len(all_encoded)

        # Update pi: average of gamma at t=0 across all sequences
        self.pi = np.sum([all_gamma[r][:, 0] for r in range(R)], axis=0) / R

        # Update A: sum of xi over time and sequences, normalize each row
        A_num = np.sum([all_xi[r].sum(axis=2) for r in range(R)], axis=0)
        self.A = A_num / A_num.sum(axis=1, keepdims=True)

        # Update B: expected emissions per state
        B_num = np.zeros((self.n_states, self.n_symbols))
        for r in range(R):
            T = len(all_encoded[r])
            for k in range(self.n_symbols):
                mask = np.array(all_encoded[r]) == k
                B_num[:, k] += all_gamma[r][:, mask].sum(axis=1)

        self.B = B_num / B_num.sum(axis=1, keepdims=True)


    def baum_welch(self, sequences, max_iter=100, epsilon=1e-4):
        """
        Trains the HMM using the Baum-Welch algorithm over multiple sequences.

        Parameters
        ----------
        sequences : list of list of int
            List of encoded sequences to train on.
        max_iter : int, optional
            Maximum number of iterations (default: 100).
        epsilon : float, optional
            Convergence threshold for log-likelihood (default: 1e-4).
    
        Returns
        -------
        log_likelihoods : list of float
            Log-likelihood at each iteration, useful for diagnostics.
        """

        self.log_likelihoods = []

        for iteration in range(max_iter):
            all_gamma = []
            all_xi = []
            log_likelihood = 0

            # E-step: forward, backward, gamma, xi for each sequence
            for enc in sequences:
                self.forward(enc)
                self.backward(enc)
                self.compute_gamma()
                self.compute_xi(enc)

                all_gamma.append(self.gamma.copy())
                all_xi.append(self.xi.copy())
                log_likelihood += np.sum(np.log(self.G_alpha))

            self.log_likelihoods.append(log_likelihood)
            print(f"Iteration {iteration + 1} | log-likelihood: {log_likelihood:.4f}")

            # Check convergence
            if iteration > 0 and abs(self.log_likelihoods[-1] - self.log_likelihoods[-2]) < epsilon:
                print(f"Converged at iteration {iteration + 1}")
                break

            # M-step: update parameters
            self._update_parameters(all_gamma, all_xi, sequences)

        return self.log_likelihoods

    def evaluate(self, encoded_seq, true_labels):
    
        self.forward(encoded_seq)
        self.backward(encoded_seq)
        self.compute_gamma()
    
        T = len(encoded_seq)
        true_labels = np.array(true_labels)  # ensure numpy array

        # 1. Cross-entropy — how confident is the model on the true labels
        ce = -np.mean(np.log(self.gamma[true_labels, np.arange(T)]))

        # 2. Per-residue accuracy — how often did we predict the right state
        predicted = np.argmax(self.gamma, axis=0)
        accuracy = np.mean(predicted == true_labels)

        return ce, accuracy

## Example with Casino

### Generate Samples

2 states loaded and fair dice

- A : np.ndarray of shape (n_states, n_states) - Transition probability matrix, where A[i][j] is the probability of transitioning from state i to state j.
    
- B : np.ndarray of shape (n_states, n_symbols) - Emission probability matrix, where B[i][k] is the probability of emitting symbol k in state i.

In [5]:
A_test = np.array([[0.9, 0.1], [0.23, 0.77]])
B_test = np.array([[1/6, 1/6, 1/6, 1/6, 1/6, 1/6],
                   [1/10, 1/10, 1/10, 1/10, 1/10, 1/2]])

np.random.seed(42)
sizes_test = np.random.randint(100, high=200, size=100, dtype=int)

alphabet = "123456"
sequences = []
labels = []

for size in sizes_test:
    sequence = []
    label = []

    # start in a random state according to uniform initial distribution
    state = np.random.choice(2)

    for t in range(size):
        # store the true state
        label.append(state)

        # emit a symbol according to B[state]
        symbol = np.random.choice(len(alphabet), p=B_test[state])
        sequence.append(symbol)

        # transition to next state according to A[state]
        state = np.random.choice(2, p=A_test[state])

    sequences.append(sequence)
    labels.append(label)

# quick sanity check
print(f"Example sequence: {sequences[0][:10]}")
print(f"Example labels:   {labels[0][:10]}")
print(f"State 0 = fair die, State 1 = loaded die")

Example sequence: [5, 2, 1, 2, 1, 1, 4, 5, 5, 2]
Example labels:   [1, 1, 1, 0, 0, 0, 0, 0, 0, 0]
State 0 = fair die, State 1 = loaded die


### Implement model and train with Baum-Welch

In [6]:
states = 2
alphabet="123456"

hmm_casino = HMM_efficient(states, alphabet, seed = 42)
log_likelihoods = hmm_casino.baum_welch(sequences, max_iter=1000)

Iteration 1 | log-likelihood: -28240.1027
Iteration 2 | log-likelihood: -26509.2782
Iteration 3 | log-likelihood: -26500.1891
Iteration 4 | log-likelihood: -26493.7439
Iteration 5 | log-likelihood: -26489.8751
Iteration 6 | log-likelihood: -26487.8306
Iteration 7 | log-likelihood: -26486.7997
Iteration 8 | log-likelihood: -26486.2631
Iteration 9 | log-likelihood: -26485.9633
Iteration 10 | log-likelihood: -26485.7841
Iteration 11 | log-likelihood: -26485.6721
Iteration 12 | log-likelihood: -26485.6002
Iteration 13 | log-likelihood: -26485.5535
Iteration 14 | log-likelihood: -26485.5230
Iteration 15 | log-likelihood: -26485.5028
Iteration 16 | log-likelihood: -26485.4895
Iteration 17 | log-likelihood: -26485.4807
Iteration 18 | log-likelihood: -26485.4747
Iteration 19 | log-likelihood: -26485.4707
Iteration 20 | log-likelihood: -26485.4679
Iteration 21 | log-likelihood: -26485.4659
Iteration 22 | log-likelihood: -26485.4645
Iteration 23 | log-likelihood: -26485.4633
Iteration 24 | log-l

In [7]:
print(np.round(hmm_casino.A, 2))

[[0.75 0.25]
 [0.1  0.9 ]]


In [8]:
print(np.round(hmm_casino.B, 2))

[[0.09 0.08 0.12 0.09 0.1  0.52]
 [0.17 0.17 0.16 0.17 0.16 0.17]]


This is the **label switching problem**. Baum-Welch has no way of knowing which state index corresponds to which real state, it just finds the best parameters for whatever arbitrary labeling it started with.

## Evaluate Performance

This is where Viterbi can come into play!

Viterbi finds the single most likely path through the sequence. It's great for producing a concrete prediction but it's a hard assignment,  every position gets exactly one state with no uncertainty.

Posterior decoding (which you already have via γ\gamma
γ) is often better for evaluation because:

It gives a probability distribution over states at each position
You can threshold or pick the most likely state per position independently
It tends to have higher per-residue accuracy than Viterbi

In [9]:
# flip predicted labels
flipped_labels = [1 - l for l in labels[1]]
cross_entropy, accuracy = hmm_casino.evaluate(sequences[1], flipped_labels)
print(f"acc: {accuracy:.2f}")
print(f"cross entropy: {cross_entropy:.2f}")

acc: 0.71
cross entropy: 0.55
